# Chatbot Interface 
This notebook has two parts:

A) Creates Batch Replay Script:
Takes in a batch replay script (a fixed set of test prompts) and for every turn outputs:
* the assistant’s response
* the chunks retrieved from the vector store

These logs are later used for quantitative and qualitative evaluation in the Evaluation Notebook.

B) Interactive Chatbot Interface (Baseline Prototype):
Creates a simple gradio interface that allows users to interact with the baking RAG system. 
It essentially meets all the requirements of criteria 2 from the "Conversational AI Module
Assessment Description":
* Uses an LLM and has an interface that supports multi-turn dialogue (can "remember" what the user previous said within that conversation).
* All conversations are recorded/logged in a structured JSON format for reproducibility. The logs include:
  - User inputs  
  - System responses  
  - Timestamps  
  - Retrieved chunk IDs for that response  
  
  The retrieved chunk ID was an extra item that was added (not in project brief) to increase transparency and help with user based evaluation (the user study). 

**Note on System Prompt**: The system prompt used for the batch replay test script is different from the one used in the interactive Gradio chatbot. This was intentional.

The batch replay test script is to evaluate the effects of different chunking methods. Thus, the LLM is instructed to use only the retrieved RAG chunks when answering (unless greetings or to redirect users who ask non-baking questions). This ensures that the evaluation isolates the effect of chunking and does not mix in the model’s general knowledge.

For real user interaction the LLM must use internal knowledge because the knowledge base is intentionally small (a larger one would exceed token limits). Additionally, different prompt variants were needed as part of the design/ethics experiments.


In [75]:
#Imports needed:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings #previously tested google embedding and llm but ran out of credits
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
import gradio as gr
from dotenv import load_dotenv
import json  
from datetime import datetime 
import os 
import json
from rouge_score import rouge_scorer
from collections import defaultdict
import pandas as pd
from openai import OpenAI


In [76]:
#Loading the api keys from .env file
load_dotenv() 

#Defining the embedding model and LLm model used:
##Loading the same embedding model used to create knowledge base
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small"   
)
##The LLM used, this was the cheapest one
llm = ChatOpenAI(model="gpt-5-nano")

#Loading in the chroma database
DATA_PATH = r"data"
CHROMA_PATH = r"chroma_db"
#Testing out different collections = different chunking methods
#Run different collection names according to the one being tested
#Chunk_method ="cooking_chunks_fixed"
#Chunk_method ="cooking_chunks_markdown"
#Chunk_method ="cooking_chunks_semantic"

COLLECTION_NAME = "cooking_chunks_markdown"

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings_model,   
    persist_directory=CHROMA_PATH
)

#When you created the collection with collection_metadata={"hnsw:space": "cosine"}, ChromaDB saved this configuration in the collection's metadata permanently


#This is the fixed number of chunks retrieved across all different chunking methods
number_of_chunks = 4

In [61]:
#Extra check for cosine similarity:
print(f"\n✓ Collection loaded: {COLLECTION_NAME}")

#Verify the correct collection was loaded
collection = vector_store._collection
print(f"✓ Total chunks: {collection.count()}")
print(f"✓ Metadata: {collection.metadata}")

#Checking similarity metric
if collection.metadata.get('hnsw:space') == 'cosine':
    print(f"✓ CONFIRMED: Using COSINE similarity")
else:
    print(f"✗ WARNING: Using {collection.metadata.get('hnsw:space', 'l2 (default)')}")


✓ Collection loaded: cooking_chunks_fixed
✓ Total chunks: 103
✓ Metadata: {'hnsw:space': 'cosine'}
✓ CONFIRMED: Using COSINE similarity


## A) Generating Batch Replay Script

In [62]:
#This is a helper method because the batch replay output should not only contain which chunks were retrieved for that user query but also
#What their score was

def retrieve_chunks_with_scores_and_docs(query: str, k: int = 4):
    """
    Input: query is users statment, K is the number of chunks returned 
    Output: JSON like format with chunk id's and their scores + an array with the documents and their scores (what imilarity_search_with_relevance_scores outputs)
    e.g
      retrieved_context: [
        {"chunk_id": "0", "similarity_score": 0.92},
        {"chunk_id": "3", "similarity_score": 0.88},
        ...
      ]
      docs_with_scores: [(doc, score), ...]
    """
    #This function returns the chunk and its similiarity score measured by cosine similairity 
    docs_with_scores = vector_store.similarity_search_with_relevance_scores(query, k=k)

    retrieved_context = []
    for doc, score in docs_with_scores:
        #for each chunk and it score
        cid = doc.metadata.get("chunk_id")
        if cid is not None:
            #Get the chunk id and it's score into the json like format {}
            retrieved_context.append({
                "chunk_id": str(cid),
                "similarity_score": round(float(score), 2)
            })
    return retrieved_context, docs_with_scores


In [63]:

#The input is a JSON file that has different multi-turn dialogues, the users query, the ideal outputted response, and the ideal chunk or chunks that should be returned

def batch_replay_rag(input_file: str, output_file: str = None):
    """
    Input: JSON test file
    Output: Produces a batch replay JSON file including:
      - user_utterance
      - ground_truth (ideal_assistant utterance)
      - system_response (LLM output)
      - ideal_chunk (gold chunk from test script)
      - retrieved_context: [{chunk_id, similarity_score}]
      - timestamp, meta
    Conversation memory is kept via a string conversation_history included in the RAG prompt,
    matching your interface pattern.
    """

    #Opens JSON test file
    with open(input_file, "r", encoding="utf-8") as f:
        test_dialogues = json.load(f)

    #Start of returned JSON file
    all_results = []

    #For each dialogue within all the test dialogues
    for d_idx, dialogue in enumerate(test_dialogues, start=1):
        #Debugging helper
        print(f"\n--- Running Dialogue {d_idx} ---\n")

        dialogue_result = []
        #Conversation history stores what is being said in that dialogue, so that LLM can follow previous convos
        conversation_history = ""  

        #For each turn within the conversation
        for turn_idx, turn in enumerate(dialogue, start=1):
            speaker = turn["speaker"].lower()

            #When the user speaks:
            if speaker == "user":
                user_text = turn["utterance"]

                #extract ground truth + ideal chunk from the test file
                ground_truth = None
                ideal_chunk = None

                #Checking that the dialogue is not finished
                if (
                    turn_idx < len(dialogue)
                    and dialogue[turn_idx]["speaker"].lower() == "ideal_assistant"
                ):
                    #Getting the ideal assistant response
                    ground_truth = dialogue[turn_idx]["utterance"]
                    ideal_chunk = dialogue[turn_idx].get("ideal_chunk", None)

                #RAG retrieval but getting the scores
                retrieved_context, docs_with_scores = retrieve_chunks_with_scores_and_docs(
                    user_text,
                    k=number_of_chunks #stated at the top
                )

                knowledge = ""
                for doc, score in docs_with_scores:
                    knowledge += doc.page_content + "\n\n"

                #Build RAG prompt WITH conversation history
                rag_prompt = f"""
                You are a friendly and knowledgeable baking assistant.

BAKING QUESTIONS (STRICT RULES):
- You MUST base your answer ONLY on the information present in the retrieved knowledge below.
- You may NOT add, infer, guess, or complete any missing baking steps, techniques, or facts.
- When giving instructions, you MUST stop at the last step that actually appears in the retrieved text. 
  Do NOT continue the sequence. 
- If only partial method steps are available, give ONLY those steps and then state clearly that the remaining steps are not present.
- If the knowledge base does not contain enough information to answer the question fully, reply with:
  “Sorry, I don’t have that information in my baking knowledge.”

NON-BAKING QUESTIONS:
- For greetings or unrelated small talk, respond naturally and briefly.
- NEVER provide factual information about non-baking topics.

STYLE RULES:
- Keep responses clear, concise, and strictly grounded in the retrieved chunks.
- You may paraphrase, but may NOT introduce new information.
- ABSOLUTE RULE: Never hallucinate or infer ANY baking facts not present in the retrieved text.

User question: {user_text}

Conversation history: {conversation_history}

Knowledge base (ordered from most relevant to least relevant):
{knowledge}

Your response:
"""

                #LLM Response (single-prompt style; system and RAG prompt in one above)
                result = llm.invoke(rag_prompt)
                # The LLM returns a message object with .content
                system_response = result.content if hasattr(result, "content") else str(result)

                #Update conversation history with THIS model's behaviour (so the model has context over the whole dialogue)
                conversation_history += f"User: {user_text}\nAssistant: {system_response}\n"

                #Save turn result
                dialogue_result.append({
                    "user_utterance": user_text, #what user said
                    "ground_truth": ground_truth, #ideally should have been replied
                    "system_response": system_response,   #what was replied
                    "ideal_chunk": ideal_chunk,           #the ideal chunk retrieved for that reply, if non than the test will have NA
                    "retrieved_context": retrieved_context, #the score and chunks retrieved for that query
                    "meta": {
                        "action": {"type": "generation"}
                    },
                    "timestamp": datetime.now().isoformat() + "Z" #the timestamp; important for logging
                })

                #to ensure method is working correctly
                print(f"User: {user_text}")
                print(f"LLM: {system_response}")
                print(f"Retrieved: {retrieved_context}\n")
        #Next dialgoue to explore, append current one info
        all_results.append({
            "dialogue_id": d_idx,
            "turns": dialogue_result
        })

        #Test to ensure method is working as it should
        print(f"--- End of Dialogue {d_idx} ---")

    #Save output in project folder
    os.makedirs("logs", exist_ok=True)

    #The outputted JSON file will be inside the LOGS folder and will have the timestamp on it
    if output_file is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_file = f"logs/rag_batch_output_{timestamp}.json"

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=4, ensure_ascii=False)
    #test
    print(f"\nBatch replay saved to: {output_file}\n")
    return output_file

In [64]:
#Running the batch replay for a test prompt JSON script:
batch_replay_rag("Batch_ test_2_specific.json")


--- Running Dialogue 1 ---

User: Do you know any cookie recipes? Just give me an overview of the ones you know?
LLM: Here’s the cookie recipe I know from the knowledge base:

- Chocolate chip cookies — Description: Delicious chewy chocolate chip cookies that don't require chilling, and are perfectly soft, chewy and crunchy at the same time! 
  - Prep Time: 20 minutes
  - Cook Time: 24 minutes
  - Cooling Time: 30 minutes
  - Total Time: 1 hour 14 minutes
  - Servings: 20 cookies

Ingredients:
- 175 g unsalted butter
- 225 g light brown soft sugar
- 100 g white granulated sugar
- 1 tsp vanilla extract
- 1 medium egg
- 1 egg yolk
- 275 g plain flour
- 1/2 tsp bicarbonate of soda
- 1/2 tsp sea salt
- 325 g chocolate chips

If you’d like, I can help with clarifying what’s in this recipe or how the ingredients contribute to its texture.
Retrieved: [{'chunk_id': '14', 'similarity_score': 0.5}, {'chunk_id': '45', 'similarity_score': 0.47}, {'chunk_id': '48', 'similarity_score': 0.46}, {'chu

'logs/rag_batch_output_20251208_172110.json'

In [69]:
def get_chunk(chunk_id):
    """Return a document by chunk_id."""
    result = vector_store.get(ids=[str(chunk_id)])
    return result

# Example usage:
chunk = get_chunk(17)
print(chunk["documents"][0])
#print(chunk["metadatas"][0])

# Recipe chocolate chip cookies Ingredients
175 g unsalted butter
225 g light brown soft sugar
100 g white granulated sugar
1 tsp vanilla extract
1 medium egg
1 egg yolk
275 g plain flour
1/2 tsp bicarbonate of soda
1/2 tsp sea salt
325 g chocolate chips


## B) Interactive Chatbot Interface

In [ ]:
#Decided to use recursive chunking:

#Re-initalising to ensure no confounds with other cells
COLLECTION_NAME = "cooking_chunks_fixed"

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings_model,   
    persist_directory=CHROMA_PATH
)

number_of_chunks = 4
#Extra check for cosine similarity:
print(f"\n✓ Collection loaded: {COLLECTION_NAME}")

#Verify the correct collection was loaded
collection = vector_store._collection
print(f"✓ Total chunks: {collection.count()}")
print(f"✓ Metadata: {collection.metadata}")

#Checking similarity metric
if collection.metadata.get('hnsw:space') == 'cosine':
    print(f"✓ CONFIRMED: Using COSINE similarity")
else:
    print(f"✗ WARNING: Using {collection.metadata.get('hnsw:space', 'l2 (default)')}")

In [ ]:
################################# Logging Conversation ##########
#This creates a folder called logged_current_convo
#Every new conversation gets a separate log

LOG_FOLDER = "logged_current_convo"
os.makedirs(LOG_FOLDER, exist_ok=True)

#Global placeholders that will reset as conversations change
SESSION_ID = None
LOG_FILENAME = None

SESSION_LOG = None

def init_new_session():
    """Creates a new session ID, filename, and cleared log object."""
    global SESSION_ID, LOG_FILENAME, SESSION_LOG

    SESSION_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
    LOG_FILENAME = os.path.join(LOG_FOLDER, f"conversation_{SESSION_ID}.json")

    SESSION_LOG = {
        "session_id": SESSION_ID,
        "turns": []
    }

def write_log():
    """Writes the current SESSION_LOG to the session's JSON file."""
    if SESSION_LOG is None:
        return  

    with open(LOG_FILENAME, "w") as f:
        json.dump(SESSION_LOG, f, indent=4)

    print(f"\n--- Conversation saved to {LOG_FILENAME} ---\n")


## Model A (Baseline prototype/ Non culturally aware)
This was the prompt used.

In [ ]:
def stream_response(message, history):
    """Function that allows the LLM to remember what the user has said + logs the conversation so far
    Input: the current message and previous conversation
    Output: outputted to gradio interface """
    #Logging that the users turn has start and their time
    if history == []:
        print(">>> NEW CONVERSATION DETECTED — starting a new log file")
        init_new_session()
    user_timestamp = datetime.now().isoformat() + 'Z'

    #Retrieve chunks + similarity scores, ordered highest to lowest similarity
    docs_with_scores = vector_store.similarity_search_with_relevance_scores(
        message,
        k=number_of_chunks #defined at top
    )

    #Building knowledge base in correct order
    knowledge = ""

    #For testing purposes:
    print("\n--- RETRIEVED CHUNKS (ORDERED BY RELEVANCE) ---")
    for i, (doc, score) in enumerate(docs_with_scores, start=1):
        print(f"Rank {i} | Score: {score}")
        print(f"Chunk Content:\n{doc.page_content}")
        print("-------------------------------------\n")

        #Append to knowledge in the SAME ORDER the LLM will see it
        knowledge += doc.page_content + "\n\n"

    #Build RAG prompt
    rag_prompt = f""" You are a friendly and knowledgeable baking assistant. Your goal is to help the user with accurate, practical baking guidance.

USE OF KNOWLEDGE SOURCES

You may use your internal baking knowledge freely for all questions.

You may optionally use the retrieved knowledge base below if it is relevant.

If you use retrieved knowledge, do not invent steps or details that aren't present.

If a recipe is incomplete in the retrieved knowledge, you may switch to your internal knowledge to fill in the gaps naturally.

BEHAVIOUR RULES

Provide clear, concise baking explanations or recipes.

Default to whatever baking knowledge you consider most appropriate.

In general be concise and friendly with your responses. 

NON-BAKING QUESTIONS

Respond politely and briefly.

Do not provide factual information outside of baking and cooking.

User Message: {message}
Conversation History: {history}
Retrieved Knowledge (optional contextual help):

{knowledge}

Your response:"""

    #Stream LLM response
    partial_message = ""
    for response in llm.stream(rag_prompt):
        partial_message += response.content
        #This allows the LLM to output the response as soon as it generates it
        #Doesn't need to output it only once it's all been generated
        yield partial_message

    #LOGGING the response
    SESSION_LOG["turns"].append({
        "timestamp": user_timestamp,
        "role": "user",
        "message": message
    })

    #Adding the time of the response
    llm_timestamp = datetime.now().isoformat() + 'Z'
    SESSION_LOG["turns"].append({
        "timestamp": llm_timestamp,
        "role": "llm_assistant",
        "message": partial_message
    })

    #Calling the function to log the information
    write_log()


In [ ]:
#########################  GRADIO APP / LLM INTERFACE 
#Initiate the Gradio app
chatbot = gr.ChatInterface(stream_response, 
    textbox=gr.Textbox(placeholder="Ready to embark on your culinary journey...",
        container=False,
        autoscroll=True,
        scale=7
    ),
    title="BakeBot: Your Personal Baking Assistant"
)

#Launch the Gradio app - set share=True if you want to access it outside your local network (did it for user's testing through zoom)
chatbot.launch(inbrowser=True, share=True, theme=gr.themes.Soft())

## Model B (Culturally aware system)
This was the rag prompt used for the second system.

In [ ]:
def stream_response(message, history):
    """Function that allows the LLM to remember what the user has said + logs the conversation so far
    Input: the current message and previous conversation
    Output: outputted to gradio interface """
    #Logging that the users turn has start and their time
    if history == []:
        print(">>> NEW CONVERSATION DETECTED — starting a new log file")
        init_new_session()
    user_timestamp = datetime.now().isoformat() + 'Z'

    #Retrieve chunks + similarity scores, ordered highest to lowest similarity
    docs_with_scores = vector_store.similarity_search_with_relevance_scores(
        message,
        k=number_of_chunks #defined at top
    )

    #Building knowledge base in correct order
    knowledge = ""

    #For testing purposes:
    print("\n--- RETRIEVED CHUNKS (ORDERED BY RELEVANCE) ---")
    for i, (doc, score) in enumerate(docs_with_scores, start=1):
        print(f"Rank {i} | Score: {score}")
        print(f"Chunk Content:\n{doc.page_content}")
        print("-------------------------------------\n")

        #Append to knowledge in the SAME ORDER the LLM will see it
        knowledge += doc.page_content + "\n\n"

    #Build RAG prompt
    rag_prompt = f""" You are a friendly, knowledgeable, and culturally inclusive baking assistant. You aim to help users with baking guidance while respecting global baking traditions.

CORE VALUES

Cultural diversity: Avoid assuming Western/European baking is the default.

Representation: When appropriate, mention or offer options from a variety of cultural traditions (African, Middle Eastern, Latin American, South Asian, East Asian, Caribbean…).

Respectful language: Describe all cuisines neutrally, without stereotypes.

Transparency: If your baking knowledge is stronger for one region than another, acknowledge this briefly.

User agency: If relevant, you may ask a brief clarifying question about the user's tastes or cultural context, BUT ONLY AFTER giving a direct answer first.



USE OF KNOWLEDGE SOURCES

You may freely use your internal baking knowledge, especially when users ask about global desserts, traditions, or culturally specific dishes.

You may optionally use the retrieved knowledge base below if it is relevant, but you are not restricted to it.

Make it clear when you are relying on internal knowledge rather than the retrieved knowledge base.

BEHAVIOUR RULES

When a user does not specify a culture, offer options from multiple global regions, not only Western defaults.

When a user does specify a region or tradition, prioritise culturally aligned suggestions.

If the retrieved knowledge is narrow or Western-centric, you may expand using internal knowledge while being transparent.

In general be concise and friendly with your responses. 

NON-BAKING QUESTIONS

Respond politely and briefly.

Do not provide factual information outside baking/cooking.

User Message: {message}
Retrieved Knowledge (optional contextual help):
{knowledge}
Conversation History: {history}

Your response:"""

    #Stream LLM response
    partial_message = ""
    for response in llm.stream(rag_prompt):
        partial_message += response.content
        #This allows the LLM to output the response as soon as it generates it
        #Doesn't need to output it only once it's all been generated
        yield partial_message

    #LOGGING the response
    SESSION_LOG["turns"].append({
        "timestamp": user_timestamp,
        "role": "user",
        "message": message
    })

    #Adding the time of the response
    llm_timestamp = datetime.now().isoformat() + 'Z'
    SESSION_LOG["turns"].append({
        "timestamp": llm_timestamp,
        "role": "llm_assistant",
        "message": partial_message
    })

    #Calling the function to log the information
    write_log()


In [ ]:
#########################  GRADIO APP / LLM INTERFACE 
#Initiate the Gradio app
chatbot = gr.ChatInterface(stream_response, 
    textbox=gr.Textbox(placeholder="Ready to embark on your culinary journey...",
        container=False,
        autoscroll=True,
        scale=7
    ),
    title="BakeBot: Your Personal Baking Assistant"
)

#Launch the Gradio app - set share=True if you want to access it outside your local network (did it for user's testing through zoom)
chatbot.launch(inbrowser=True, share=True, theme=gr.themes.Soft())

## Model Used to Test Baking Bias
Note: the same code above was used to test for westernisation bias, the only thing that was changed was that the history part of the prompt was deleted. This allowed for multiple single turn tested. A fix test of prompts was manually inserted and all conversations were recorded using logging. Then conversational were manually viewed to determine if there was westernisation.

Important to note, when deciding how many cultural regions, this is how they were grouped:
* Western/European / North American

* East Asian

* South Asian

* Middle Eastern / North African

* Sub-Saharan African

* Latin American / Caribbean

* Other / Mixed / unclear

Below is also the code used to find averages. For the averages and stasticial measures calculated for the user study, this was done through google surveys and then google sheets. 

In [88]:
#To calculate the averages
#only the first 10 questions were baking neutral questions
import json
import pandas as pd

file_path = "test_1_bias_model_A_annotated_with_metrics.json"

with open(file_path, "r") as f:
    data = json.load(f)

rows = []
for turn in data["turns"]:
    if turn["role"] == "llm_assistant":
        metrics = turn.get("metrics", {})
        rows.append({
            "western_only": metrics.get("western_only", None),
            "regions": metrics.get("regions", None),
            "ack_limitations": metrics.get("ack_limitations", None)
        })

df = pd.DataFrame(rows)

df_first10 = df.iloc[:10]

average_metrics = df_first10.mean()

print("Averages for first 10 dialogues:")
print(average_metrics)


Averages for first 10 dialogues:
western_only       0.9
regions            1.5
ack_limitations    0.0
dtype: float64


In [89]:
#To calculate the averages
#only the first 10 questions were baking neutral questions
import json
import pandas as pd

file_path = "Test_2_bias_model_b_annotated_with_metrics.json"

with open(file_path, "r") as f:
    data = json.load(f)

rows = []
for turn in data["turns"]:
    if turn["role"] == "llm_assistant":
        metrics = turn.get("metrics", {})
        rows.append({
            "western_only": metrics.get("western_only", None),
            "regions": metrics.get("regions", None),
            "ack_limitations": metrics.get("ack_limitations", None)
        })

df = pd.DataFrame(rows)

df_first10 = df.iloc[:10]

average_metrics = df_first10.mean()

print("Averages for first 10 dialogues:")
print(average_metrics)


Averages for first 10 dialogues:
western_only       0.0
regions            4.3
ack_limitations    0.0
dtype: float64
